# Compare Original And Cropped EMBED Images

This notebook compares the processed original EMBED PNG images with the cropped PNG images for one patient.

Important note:
- The notebook itself does **not** resize the original image for display.
- However, the files in `EMBED_Dataset_Split` were already resized earlier in `preprocess_img_embed.py`.
- The files in `EMBED_Split_Cropped_PNG` were also resized in `crop_img_embed.py` after cropping.

Because of that, the original and cropped saved PNGs often have the same final canvas size.

To make the crop effect easier to inspect, this notebook shows three views for each filename:
1. the saved original processed image
2. a tight crop extracted directly from that original image without padding back to the fixed canvas
3. the saved cropped output image

That middle panel is the best way to see what region was actually kept by the crop.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

## Configuration

Set the patient ID here, then run the notebook from top to bottom.

In [ ]:
ORIGINAL_ROOT = Path("/mnt/cv_data/users/mengxu/EMBED_Dataset_Split")
CROPPED_ROOT = Path("/mnt/cv_data/users/mengxu/EMBED_Split_Cropped_PNG")
SPLITS = ("train", "val", "test")

PATIENT_ID = "10141610"

## Helper Functions

These functions load the images, normalize them for display, find the breast bounding box, and collect matching original and cropped files.

In [ ]:
def load_grayscale_image(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    return image


def normalize_for_display(image):
    image = image.astype(np.float32)
    min_value = float(image.min())
    max_value = float(image.max())

    if max_value <= min_value:
        return np.zeros(image.shape, dtype=np.uint8)

    normalized = (image - min_value) / (max_value - min_value)
    return (normalized * 255).astype(np.uint8)


def normalize_patient_id(patient_id):
    patient_id = str(patient_id).strip()
    if patient_id.endswith(".0"):
        patient_id = patient_id[:-2]
    return patient_id


def extract_patient_id_from_filename(image_path):
    return normalize_patient_id(image_path.stem.split("_", 1)[0])


def find_breast_bounding_box(image):
    image_uint8 = normalize_for_display(image)
    _, binary_mask = cv2.threshold(
        image_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    contours, _ = cv2.findContours(
        binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return 0, 0, image.shape[1], image.shape[0]

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, width, height = cv2.boundingRect(largest_contour)

    if width <= 0 or height <= 0:
        return 0, 0, image.shape[1], image.shape[0]

    return x, y, width, height


def extract_tight_crop(image):
    x, y, width, height = find_breast_bounding_box(image)
    cropped = image[y:y + height, x:x + width]

    if cropped.size == 0:
        return image

    return cropped


def collect_patient_pairs(patient_id, original_root, cropped_root):
    patient_id = normalize_patient_id(patient_id)
    pairs = []

    for split_name in SPLITS:
        split_original_dir = original_root / split_name
        split_cropped_dir = cropped_root / split_name

        if not split_original_dir.exists():
            continue

        for original_path in sorted(split_original_dir.rglob("*.png")):
            if extract_patient_id_from_filename(original_path) != patient_id:
                continue

            cropped_path = split_cropped_dir / original_path.name
            pairs.append(
                {
                    "split": split_name,
                    "filename": original_path.name,
                    "original_path": original_path,
                    "cropped_path": cropped_path if cropped_path.exists() else None,
                }
            )

    return pairs


def collect_available_patient_ids(original_root, limit=10):
    patient_ids = set()

    for split_name in SPLITS:
        split_original_dir = original_root / split_name
        if not split_original_dir.exists():
            continue

        for original_path in split_original_dir.rglob("*.png"):
            patient_ids.add(extract_patient_id_from_filename(original_path))

    return sorted(patient_ids)[:limit]

## Load Matching Images

This cell gathers all original images for the selected patient across the dataset splits and matches them to cropped images with the same filename.

In [ ]:
patient_id = normalize_patient_id(PATIENT_ID)
pairs = collect_patient_pairs(patient_id, ORIGINAL_ROOT, CROPPED_ROOT)

if not pairs:
    sample_patient_ids = collect_available_patient_ids(ORIGINAL_ROOT)
    sample_text = ", ".join(sample_patient_ids) if sample_patient_ids else "none found"
    raise FileNotFoundError(
        f"No original PNG images found for patient {patient_id} under {ORIGINAL_ROOT}. "
        f"Sample patient IDs: {sample_text}"
    )

print(f"Found {len(pairs)} image pair entries for patient {patient_id}.")
for pair in pairs:
    print(f"[{pair['split']}] {pair['filename']}")

## Display The Comparison

Run this cell to show each image as three panels:
- original processed image
- tight crop extracted from the original image without padding back to the fixed canvas
- saved cropped output image

The titles also show each image shape.

In [ ]:
num_rows = len(pairs)
figure, axes = plt.subplots(
    num_rows,
    3,
    figsize=(16, max(4, 4 * num_rows)),
    squeeze=False,
)

figure.suptitle(
    f"Patient {patient_id}: original vs tight crop vs cropped output",
    fontsize=14,
    y=0.995,
)

for row_index, pair in enumerate(pairs):
    original_axis, tight_crop_axis, cropped_axis = axes[row_index]

    original_image = load_grayscale_image(pair["original_path"])
    tight_crop_image = extract_tight_crop(original_image)

    original_axis.imshow(normalize_for_display(original_image), cmap="gray")
    original_axis.set_title(
        f"Original\n{pair['split']} | {pair['filename']}\nshape={original_image.shape}",
        fontsize=10,
    )
    original_axis.axis("off")

    tight_crop_axis.imshow(normalize_for_display(tight_crop_image), cmap="gray")
    tight_crop_axis.set_title(
        f"Tight Crop From Original\nshape={tight_crop_image.shape}",
        fontsize=10,
    )
    tight_crop_axis.axis("off")

    if pair["cropped_path"] is not None:
        cropped_image = load_grayscale_image(pair["cropped_path"])
        cropped_axis.imshow(normalize_for_display(cropped_image), cmap="gray")
        cropped_axis.set_title(
            f"Saved Cropped Output\nshape={cropped_image.shape}",
            fontsize=10,
        )
    else:
        cropped_axis.text(
            0.5,
            0.5,
            "Missing cropped image",
            ha="center",
            va="center",
            fontsize=11,
        )
        cropped_axis.set_title("Saved Cropped Output", fontsize=10)

    cropped_axis.axis("off")

figure.tight_layout(rect=(0, 0, 1, 0.98))
plt.show()